# 캐글 대회

In [106]:
# 필요한 모듈 임포트
import os, hds
import pandas as pd
import numpy as np
from plt_rcs import *

In [107]:
os.getcwd()

'c:\\Users\\pc\\Desktop\\Repo\\SeSAC\\study\\sesac_ml_dl_study_repo\\kaggle_data_2\\data'

In [108]:
os.chdir('../data')

In [109]:
sorted(os.listdir())

['base_lgbm_model.csv',
 'default_model.csv',
 'ensemble_rank_avg_model.csv',
 'multi_model_ensemble_model.csv',
 'sample_submission.csv',
 'test.csv',
 'train.csv']

In [110]:
# 데이터셋 불러오기
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

## 데이터 조회

In [111]:
train.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Num ID
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure,14860
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure,47181
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure,47182
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure,47183
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure,47184


In [112]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Target                   10000 non-null  int64  
 9   Failure Type             10000 non-null  object 
 10  Num ID                   10000 non-null  int64  
dtypes: float64(3), int64(5), object(3)
memory usage: 859.5+ KB


In [113]:
train.describe().round(3)

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Num ID
count,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000
mean,5000.500,300.005,310.006,1538.776,39.987,107.951,0.034,40711.266
std,2886.896,2.000,1.484,179.284,9.969,63.654,0.181,14870.161
min,1.000,295.300,305.700,1168.000,3.800,0.000,0.000,14860.000
25%,2500.750,298.300,308.800,1423.000,33.200,53.000,0.000,23214.750
50%,5000.500,300.100,310.100,1503.000,40.100,108.000,0.000,48861.500
75%,7500.250,301.500,311.100,1612.000,46.800,162.000,0.000,53001.500
max,10000.000,304.500,313.800,2886.000,76.600,253.000,1.000,57174.000


In [114]:
train.describe(include=object)

,Product ID,Type,Failure Type
count,10000,10000,10000
unique,10000,3,6
top,M14860,L,No Failure
freq,1,6000,9652


In [115]:
train['Failure Type'].value_counts()

Failure Type
No Failure                  9652
Heat Dissipation Failure     112
Power Failure                 95
Overstrain Failure            78
Tool Wear Failure             45
Random Failures               18
Name: count, dtype: int64

In [116]:
test.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type
0,10001,M24860,M,300.5,309.5,1451,47.8,93,No Failure
1,10002,M24861,M,301.4,311.1,1697,35.6,160,No Failure
2,10003,H39413,H,304.0,313.1,1612,33.7,100,No Failure
3,10004,L57175,L,298.6,310.5,1276,55.2,91,No Failure
4,10005,L57176,L,299.9,310.8,1400,46.2,219,No Failure


In [117]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      3000 non-null   int64  
 1   Product ID               3000 non-null   object 
 2   Type                     3000 non-null   object 
 3   Air temperature [K]      3000 non-null   float64
 4   Process temperature [K]  3000 non-null   float64
 5   Rotational speed [rpm]   3000 non-null   int64  
 6   Torque [Nm]              3000 non-null   float64
 7   Tool wear [min]          3000 non-null   int64  
 8   Failure Type             3000 non-null   object 
dtypes: float64(3), int64(3), object(3)
memory usage: 211.1+ KB


## 오입력 처리

In [118]:
# 오입력 케이스B : Target=1 + No Failure → 0
train.loc[train['Failure Type'].eq('No Failure') & train['Target'].eq(1), 'Target'] = 0

In [119]:
# 오입력 케이스C : Target=0 + Random Failures → 1
# train.loc[train['Failure Type'].ne('No Failure') & train['Target'].eq(0), 'Target'] = 1

## 특성 행렬과 타겟 벡터로 분리

In [120]:
train.columns

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Target', 'Failure Type', 'Num ID'],
      dtype='object')

In [121]:
# 타겟 벡터명 설정
yvar = 'Target'

# 훈련셋 특성 행렬 생성
X = train.drop(columns = yvar)

# 훈련셋 타겟 벡터 생성
y = train[yvar].copy()

In [122]:
# 시험셋 특성 행렬 생성
X_test = test.copy()

## 데이터 전처리

In [123]:
# UID 제거 및 Product ID 인덱스 설정 (Failure Type 유지)
X = X.drop(columns=['UDI', 'Num ID']).set_index(keys='Product ID')
X_test = X_test.drop(columns=['UDI']).set_index(keys='Product ID')

In [124]:
X.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Failure Type'],
      dtype='object')

In [125]:
X_test.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Failure Type'],
      dtype='object')

In [126]:
# 컬럼명 변경
X.columns = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear', 'FType']
X_test.columns = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear', 'FType']

In [127]:
X.head()

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear,FType
Product ID,,,,,,,
M14860,M,298.1,308.6,1551,42.8,0,No Failure
L47181,L,298.2,308.7,1408,46.3,3,No Failure
L47182,L,298.1,308.5,1498,49.4,5,No Failure
L47183,L,298.2,308.6,1433,39.5,7,No Failure
L47184,L,298.2,308.7,1408,40.0,9,No Failure


In [128]:
# Type 오디널 인코딩: L=0, M=1, H=2
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[['L', 'M', 'H']])
X['Type'] = oe.fit_transform(X[['Type']])
X_test['Type'] = oe.transform(X_test[['Type']])

# Failure Type 인코딩: No Failure=0, 나머지 고장 유형별 번호
ft_categories = ['No Failure', 'Heat Dissipation Failure', 'Overstrain Failure',
                 'Power Failure', 'Random Failures', 'Tool Wear Failure']
oe_ft = OrdinalEncoder(categories=[ft_categories])
X['FType'] = oe_ft.fit_transform(X[['FType']])
X_test['FType'] = oe_ft.transform(X_test[['FType']])

X.head()

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear,FType
Product ID,,,,,,,
M14860,1.0,298.1,308.6,1551,42.8,0,0.0
L47181,0.0,298.2,308.7,1408,46.3,3,0.0
L47182,0.0,298.1,308.5,1498,49.4,5,0.0
L47183,0.0,298.2,308.6,1433,39.5,7,0.0
L47184,0.0,298.2,308.7,1408,40.0,9,0.0


## 피처 엔지니어링

In [129]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [130]:
# 파생변수 추가 함수
def add_features(df):
    df = df.copy()
    # 열 방출 능력(상관계수 0.88)
    df['temp_diff'] = df['ProcTmp'] - df['AirTmp']
    # 기계 출력(상관계수 -0.88)
    df['power'] = df['Torque'] * df['RotSpd']
    # 마모 상태에서의 부하
    df['wear_torque'] = df['ToolWear'] * df['Torque']
    # 마모 상태에서의 회전
    df['wear_speed'] = df['ToolWear'] * df['RotSpd']
    # 단위 토크당 온도차 (열 방출 효율)
    df['temp_per_torque'] = df['temp_diff'] / (df['Torque'] + 1e-6)
    # 출력 대비 온도차 비율
    df['temp_per_power'] = df['temp_diff'] / (df['power'] + 1e-6)
    
    return df

In [131]:
# 파생변수 추가
X = add_features(X)
X_test = add_features(X_test)

In [132]:
X_test.describe().round(3)

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear,FType,temp_diff,power,wear_torque,wear_speed,temp_per_torque,temp_per_power
count,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000
mean,0.488,300.058,310.029,1541.423,39.830,107.698,0.090,9.971,59687.741,4313.689,165674.887,0.289,0.000
std,0.663,2.090,1.562,185.174,10.449,65.660,0.542,1.033,10680.351,2970.106,103342.844,0.863,0.000
min,0.000,294.300,305.300,1119.000,0.200,0.000,0.000,7.200,594.200,0.000,0.000,0.114,0.000
25%,0.000,298.400,308.800,1420.000,32.600,52.000,0.000,9.200,52683.350,1870.675,80340.000,0.208,0.000
50%,0.000,300.000,310.000,1514.000,39.950,105.000,0.000,9.800,59717.200,4016.800,159742.500,0.251,0.000
75%,1.000,301.700,311.200,1625.250,47.100,162.000,0.000,10.900,67079.650,6236.825,245527.500,0.309,0.000
max,2.000,305.500,314.500,2971.000,73.400,265.000,5.000,12.600,99897.400,14637.600,549288.000,47.000,0.016


## 모델 정의

### LGBM 모델

In [133]:
# 과적합 억제를 위한 LGBM 파라미터 조정
model = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    # 과적합 억제
    num_leaves=15,           # 기본 31 → 축소 (트리 복잡도 감소)
    max_depth=5,             # 트리 깊이 제한
    min_child_samples=30,    # 리프 노드 최소 샘플 수 증가
    reg_alpha=0.1,           # L1 정규화
    reg_lambda=1.0,          # L2 정규화
    subsample=0.8,           # 행 샘플링
    colsample_bytree=0.8,    # 열 샘플링
    n_estimators=200,        # 트리 수 제한
)

### XGBoost 모델

In [134]:
# from xgboost import XGBClassifier

# model = XGBClassifier(
#     scale_pos_weight=len(y[y==0]) / len(y[y==1]),  # class_weight='balanced' 대응
#     random_state=42,
#     n_jobs=-1,
# )

### RF 모델

In [135]:
# from sklearn.ensemble import RandomForestClassifier

# model = RandomForestClassifier(
#     class_weight='balanced',
#     random_state=42,
#     n_jobs=-1,
# )

### SMOTE 모델

In [136]:
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline

# # SMOTE 적용 모델 (class_weight 제거)
# model_smote = LGBMClassifier(random_state=42, n_jobs=-1)

# model = ImbPipeline([
#     ('smote', SMOTE(random_state=42)),
#     ('model', model_smote),
# ])

In [137]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, average_precision_score
from sklearn.model_selection import StratifiedKFold

def robust_cv_evaluate(model, X, y, n_seeds=10, n_splits=5):
    """
    다중 시드 × Stratified K-Fold 교차검증으로 일반화 성능 평가
    - 시드마다 다른 폴드 분할 → 시드 의존성 제거
    - AUC 외에 F1, Recall, Precision, PR-AUC 추가
    - Fold간 편차 + 시드간 편차 → 과적합 징후 탐지
    """
    seed_results = []
    
    for seed in range(n_seeds):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_metrics = {'auc': [], 'f1': [], 'recall': [], 'precision': [], 'pr_auc': []}
        
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            model.fit(X_tr, y_tr)
            y_prob = model.predict_proba(X_val)[:, 1]
            y_pred = model.predict(X_val)
            
            fold_metrics['auc'].append(roc_auc_score(y_val, y_prob))
            fold_metrics['f1'].append(f1_score(y_val, y_pred))
            fold_metrics['recall'].append(recall_score(y_val, y_pred))
            fold_metrics['precision'].append(precision_score(y_val, y_pred, zero_division=0))
            fold_metrics['pr_auc'].append(average_precision_score(y_val, y_prob))
        
        seed_results.append({k: np.mean(v) for k, v in fold_metrics.items()})
    
    # 결과 집계
    results_df = pd.DataFrame(seed_results)
    
    print("=" * 60)
    print(f"다중 시드 교차검증 결과 ({n_seeds}시드 × {n_splits}폴드)")
    print("=" * 60)
    print(f"{'지표':<12} {'평균':>8} {'표준편차':>8} {'최소':>8} {'최대':>8}")
    print("-" * 60)
    for col in results_df.columns:
        vals = results_df[col]
        print(f"{col:<12} {vals.mean():>8.5f} {vals.std():>8.5f} {vals.min():>8.5f} {vals.max():>8.5f}")
    
    # 과적합 경고
    print("-" * 60)
    auc_std = results_df['auc'].std()
    f1_std = results_df['f1'].std()
    if auc_std > 0.015:
        print("⚠ AUC 시드간 편차 큼 → 모델이 데이터 분할에 민감 (과적합 징후)")
    if results_df['pr_auc'].mean() < results_df['auc'].mean() - 0.15:
        print("⚠ PR-AUC << ROC-AUC → 불균형 데이터에서 AUC가 과대평가될 수 있음")
        print("  → PR-AUC가 캐글 점수와 더 가까울 가능성 높음")
    
    return results_df

# 실행
cv_results = robust_cv_evaluate(model, X, y, n_seeds=10, n_splits=5)

다중 시드 교차검증 결과 (10시드 × 5폴드)
지표                 평균     표준편차       최소       최대
------------------------------------------------------------
auc           1.00000  0.00000  0.99999  1.00000
f1            0.99716  0.00147  0.99551  0.99850
recall        1.00000  0.00000  1.00000  1.00000
precision     0.99441  0.00288  0.99113  0.99701
pr_auc        0.99995  0.00008  0.99981  1.00000
------------------------------------------------------------


## Train/Test 분포 비교

In [138]:
# Train/Test 피처 분포 비교 (KS 검정)
from scipy.stats import ks_2samp

print(f"{'피처':<15} {'KS 통계량':>10} {'p-value':>12} {'분포 차이':>10}")
print("-" * 50)
for col in X.columns:
    stat, pval = ks_2samp(X[col], X_test[col])
    flag = "⚠ 유의" if pval < 0.05 else "OK"
    print(f"{col:<15} {stat:>10.4f} {pval:>12.4f} {flag:>10}")

피처                  KS 통계량      p-value      분포 차이
--------------------------------------------------
Type                0.0063       1.0000         OK
AirTmp              0.0308       0.0248       ⚠ 유의
ProcTmp             0.0220       0.2095         OK
RotSpd              0.0374       0.0030       ⚠ 유의
Torque              0.0271       0.0657         OK
ToolWear            0.0280       0.0527         OK
FType               0.0024       1.0000         OK
temp_diff           0.0337       0.0103       ⚠ 유의
power               0.0181       0.4313         OK
wear_torque         0.0272       0.0646         OK
wear_speed          0.0282       0.0504         OK
temp_per_torque     0.0221       0.2053         OK
temp_per_power      0.0166       0.5455         OK


## 피처 분석 및 선택

In [139]:
# 1. LGBM 피처 중요도 (전체 train으로 학습)
model.fit(X, y)

importance_df = pd.DataFrame({
    'feature': model.feature_names_in_,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

print("=== LGBM 피처 중요도 (split 기준) ===")
print(importance_df.to_string(index=False))
print()

# 2. 순열 중요도 (CV 기반 - 과적합 방지)
from sklearn.inspection import permutation_importance

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
perm_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model.fit(X_tr, y_tr)
    r = permutation_importance(model, X_val, y_val, scoring='roc_auc', n_repeats=10, random_state=42)
    perm_scores.append(r.importances_mean)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'perm_importance': np.mean(perm_scores, axis=0),
}).sort_values('perm_importance', ascending=False)

print("=== 순열 중요도 (CV 검증셋 기준) ===")
print(perm_df.to_string(index=False))
print()

# 3. 피처 상관관계 확인 (다중공선성)
print("=== 피처간 상관계수 (|r| > 0.7만 표시) ===")
corr = X.corr()
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f"  {corr.index[i]} ↔ {corr.columns[j]}: {corr.iloc[i, j]:.3f}")

=== LGBM 피처 중요도 (split 기준) ===
        feature  importance
          FType         371
          power          94
       ToolWear          88
      temp_diff          87
    wear_torque          77
         RotSpd          61
temp_per_torque          51
         Torque          46
     wear_speed          45
        ProcTmp          42
         AirTmp          29
 temp_per_power          23
           Type           1

=== 순열 중요도 (CV 검증셋 기준) ===
        feature  perm_importance
          FType     4.290441e-02
       ToolWear     5.327317e-06
      temp_diff     1.566858e-06
          power     2.220446e-18
    wear_torque     2.220446e-18
temp_per_torque     2.220446e-18
     wear_speed     2.220446e-18
 temp_per_power     2.220446e-18
         Torque     0.000000e+00
           Type     0.000000e+00
         RotSpd     0.000000e+00
        ProcTmp     0.000000e+00
         AirTmp     0.000000e+00

=== 피처간 상관계수 (|r| > 0.7만 표시) ===
  AirTmp ↔ ProcTmp: 0.876
  RotSpd ↔ Torque: -0.875
 

## 전체 훈련셋으로 재학습 및 제출 파일 생성

In [140]:
# 다중 모델 × 다중 시드 × K-Fold 앙상블 (Rank Averaging)
from scipy.stats import rankdata
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

n_seeds = 50
n_splits = 5

# 모델 정의 함수 (시드별로 생성)
def get_models(seed):
    pos_weight = len(y[y==0]) / len(y[y==1])
    return {
        'lgbm': LGBMClassifier(
            class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1,
            num_leaves=15, max_depth=5, min_child_samples=30,
            reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8,
            n_estimators=200,
        ),
        'xgb': XGBClassifier(
            scale_pos_weight=pos_weight, random_state=seed, n_jobs=-1,
            max_depth=4, min_child_weight=30, reg_alpha=0.1, reg_lambda=1.0,
            subsample=0.8, colsample_bytree=0.8, n_estimators=200,
            verbosity=0,
        ),
        'rf': RandomForestClassifier(
            class_weight='balanced', random_state=seed, n_jobs=-1,
            max_depth=8, min_samples_leaf=20, n_estimators=200,
        ),
    }

# 모델별 예측 수집
model_preds = {name: [] for name in ['lgbm', 'xgb', 'rf']}

for seed in range(n_seeds):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed)
    
    for name, m in models.items():
        fold_preds = np.zeros(len(X_test))
        for train_idx, val_idx in skf.split(X, y):
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            m.fit(X_tr, y_tr)
            fold_preds += m.predict_proba(X_test)[:, 1] / n_splits
        
        ranked = rankdata(fold_preds) / len(fold_preds)
        model_preds[name].append(ranked)

# 모델별 평균 순위
for name in model_preds:
    model_preds[name] = np.mean(model_preds[name], axis=0)
    print(f"{name} 예측 범위: {model_preds[name].min():.4f} ~ {model_preds[name].max():.4f}")

# 최종 블렌딩 (동일 가중치)
y_test_prob = np.mean(list(model_preds.values()), axis=0)

print(f"\n앙상블 완료: 3모델 × {n_seeds}시드 × {n_splits}폴드 = {3 * n_seeds * n_splits}개 모델")

# 제출 파일 생성
submit = pd.read_csv('sample_submission.csv')
submit[yvar] = y_test_prob

key = 'ftype_multi_ensemble'
submit_filename = f'{key}_model.csv'
submit.to_csv(path_or_buf=submit_filename, index=False)
print(f"제출 파일: {submit_filename}")

lgbm 예측 범위: 0.0010 ~ 1.0000
xgb 예측 범위: 0.0698 ~ 0.9985
rf 예측 범위: 0.0697 ~ 1.0000

앙상블 완료: 3모델 × 50시드 × 5폴드 = 750개 모델
제출 파일: ftype_multi_ensemble_model.csv
